# Pair-score smoke test on Colab — expert = `Ebi`

Trains `PairScoreHead` (a 2-layer MLP) on top of the transformer_v1 encoder stack (`FleetEncoder` + `PlanetEncoder` + `PlanetEntityEncoder` + `CrossEntityAttention`). Joint cross-entropy on flattened `(P×P)` pair logits, acted rows only, **expert: `Ebi`** — current Kaggle leaderboard rank 3 with the largest replay corpus available locally (434 episodes):

| player | replays | LB rank | notes |
|---|---:|---:|---|
| **Ebi** | **434** | **3** | most data + top-3 strength |
| flg | 411 | 2 | similar volume |
| Orbital Occle | 238 | — | older sample |
| bowwowforeach | 121 | 1 | small but #1 |
| kovi | 119 | — | 48.7 % winrate sample |
| Shun_PI | 117 | — | older sample |
| Erfan Eshratifar | 73 | — | small sample |

**Stages**
1. **Experiment 1** — tiny-overfit gate (50 rows, train==val).
2. **Experiment 2** — small-real split (5 000 rows, 80/20).
3. **Experiment 3** — joint fine-tune: resume from Exp 2's `pair_score_best.pt` and **unfreeze layer 2 (CrossEntityAttention)** alongside the head.

**Prerequisites in `gs://orbit-wars-shipping/`:** `code.tgz`, `data.tgz`, `weights.tgz`, `pair_score_assets.tgz`. Build and upload locally with:
```bash
PAIR_SCORE_PLAYER=Ebi INCLUDE_PAIR_SCORE_ASSETS=1 UPLOAD=1 ./scripts/pack_for_gpu.sh
```
`data.tgz` must contain Ebi's action / planet / fleet / entity / cross_entity CSVs. `pair_score_assets.tgz` carries the action-stage encoder checkpoint and `data/replays/Ebi/` for player-to-CSV matching.

**Runtime:** Runtime → Change runtime type → **T4 GPU** (or higher).

Total runtime: ~3 min setup + ~2 min Exp 1 + ~10 min Exp 2 + ~5 min Exp 3 ≈ 20 min.

## 1. Verify GPU

In [ ]:
import torch, sys
if not torch.cuda.is_available():
    sys.exit('No GPU runtime — Runtime → Change runtime type → T4 GPU, then re-run.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')

## 2. Authenticate to GCP & pull tarballs

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT = 'analog-receiver-489214-e9'
BUCKET  = 'gs://orbit-wars-shipping'
PLAYER  = 'Ebi'    # expert (current LB rank 3, 434 replays)

!gcloud config set project {PROJECT}

In [ ]:
import os
WORK = '/content/orbit-wars'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

for name in ('code.tgz', 'data.tgz', 'weights.tgz', 'pair_score_assets.tgz'):
    !gsutil cp {BUCKET}/{name} .

## 3. Unpack + install

In [ ]:
%cd {WORK}
!tar xzf code.tgz
!tar xzf data.tgz
!tar xzf weights.tgz
!tar xzf pair_score_assets.tgz

# Pair-score needs action CSVs from data.tgz, plus an action-stage
# encoder checkpoint and data/replays/<player>/ from pair_score_assets.tgz.
# Build and upload all four tarballs locally with:
#
#   PAIR_SCORE_PLAYER=Ebi INCLUDE_PAIR_SCORE_ASSETS=1 UPLOAD=1 ./scripts/pack_for_gpu.sh

import glob
from pathlib import Path

required = (
    ('data/datasets/action', 'action_*.csv', 'action CSVs'),
    ('data/datasets/planet', 'planet_*.csv', 'planet CSVs'),
    ('data/datasets/fleet', 'fleet_*.csv', 'fleet CSVs'),
    ('data/datasets/entity', 'entity_*.csv', 'entity CSVs'),
    ('data/datasets/cross_entity', 'cross_entity_*.csv', 'cross-entity CSVs'),
)
for rel, pattern, label in required:
    count = len(list(Path(rel).glob(pattern)))
    print(f'{label}: {count}')
    if count == 0:
        raise SystemExit(
            f'no {label} found under {rel}; the bucket data.tgz is stale or incomplete. '
            'Re-run scripts/pack_for_gpu.sh locally and upload the new data.tgz.'
        )

act = sorted(glob.glob('data/runs/action/*/action_best.pt'))
if not act:
    act = sorted(glob.glob('data/runs/action/*/action_last.pt'))
if not act:
    raise SystemExit(
        'no action_*.pt found — upload pair_score_assets.tgz to '
        f'{BUCKET} (build it with scripts/pack_for_gpu.sh).'
    )
ENCODER_CKPT = act[-1]
print('encoder ckpt:', ENCODER_CKPT)

# Confirm the configured PLAYER's replays are present so --player resolves.
player_replays = sorted(glob.glob(f'data/replays/{PLAYER}/*.json.gz'))
if not player_replays:
    raise SystemExit(
        f'no replays under data/replays/{PLAYER}/ — '
        'pair_score_assets.tgz must include data/replays/<player>/.'
    )
print(f'replays for {PLAYER}: {len(player_replays)}')

In [ ]:
%cd {WORK}
!pip install -q -r requirements.txt --no-deps
!pip install -q kaggle-environments

## 4. Sanity-check imports + dataset coverage

In [ ]:
import sys
sys.path.insert(0, WORK)

from agents.transformer_v1.pretrain.pair_score import (
    PairScoreHead, PairScoreStack, compute_pair_score_loss,
    discover_action_csvs, load_frozen_encoder_stack, player_replay_stems,
)
from agents.transformer_v1.pretrain.expert_action import ActionSnapshotDataset
from agents.transformer_v1.paths import (
    ACTION_DATASET_DIR, PLANET_DATASET_DIR, FLEET_DATASET_DIR,
    ENTITY_DATASET_DIR, CROSS_ENTITY_DATASET_DIR,
)
from pathlib import Path
import os, shlex, subprocess

REPLAY_DIR = Path('data/replays')
all_action_csvs = sorted(Path(ACTION_DATASET_DIR).glob('action_*.csv'))
expert_stems = player_replay_stems(REPLAY_DIR, PLAYER)
expert_csvs  = discover_action_csvs(
    Path(ACTION_DATASET_DIR), filter_mode='all',
    player=PLAYER, replay_dir=REPLAY_DIR,
)
expert_winner_csvs = discover_action_csvs(
    Path(ACTION_DATASET_DIR), filter_mode='winner',
    player=PLAYER, replay_dir=REPLAY_DIR,
)
print(f'action CSVs total:                  {len(all_action_csvs)}')
print(f'replays for {PLAYER}:                 {len(expert_stems)}')
print(f'action CSVs for {PLAYER} (all):       {len(expert_csvs)}')
print(f'action CSVs for {PLAYER} (winner):    {len(expert_winner_csvs)}')
if not expert_csvs:
    raise SystemExit(
        f'no action CSVs matched player={PLAYER!r}. This usually means '
        'data.tgz in the bucket was built before data/datasets/action existed, '
        'or pair_score_assets.tgz contains a different data/replays/<player> tree.'
    )

# Coverage check — stems present across all 5 CSV dirs.
stems = {p.stem.removeprefix('action_') for p in expert_csvs}
for d, prefix in (
    (PLANET_DATASET_DIR, 'planet_'), (FLEET_DATASET_DIR, 'fleet_'),
    (ENTITY_DATASET_DIR, 'entity_'), (CROSS_ENTITY_DATASET_DIR, 'cross_entity_'),
):
    stems &= {p.stem.removeprefix(prefix) for p in Path(d).glob(f'{prefix}*.csv')}
print(f'{PLAYER} CSVs covered by all 5 dirs: {len(stems)}')
if not stems:
    raise SystemExit('no player action CSVs are covered by all feature dirs')

def _work_path(pathlike):
    p = Path(pathlike)
    return p if p.is_absolute() else Path(WORK) / p

def run_pair_score(*, out_dir, batch_size, lr, epochs,
                   max_rows=None, overfit=False, val_frac=None,
                   init_from=None, unfreeze=None, device='cuda'):
    """Spawn the pair-score trainer with unbuffered stdout so per-epoch
    progress lines stream into the Colab cell in real time.

    ``max_rows=None`` drops the cap and trains on every acted snapshot
    available for ``PLAYER``.
    """
    encoder = _work_path(ENCODER_CKPT)
    if not encoder.exists():
        raise FileNotFoundError(f'ENCODER_CKPT does not exist: {encoder}')
    # `python -u` forces line-buffered stdout/stderr for the child so
    # the parent (Jupyter) sees each print() immediately. Without it,
    # the child's stdout block-buffers (~4 KB) and progress only
    # surfaces at the very end of the run.
    cmd = [
        sys.executable, '-u', '-m', 'agents.transformer_v1.pretrain.pair_score',
        '--encoder-ckpt', str(encoder),
        '--player', PLAYER, '--filter', 'all',
        '--batch-size', str(batch_size), '--lr', str(lr), '--epochs', str(epochs),
        '--device', device,
        '--out-dir', str(_work_path(out_dir)),
    ]
    if max_rows is not None:
        cmd += ['--max-rows', str(max_rows)]
    if overfit:
        cmd.append('--overfit')
    if val_frac is not None:
        cmd += ['--val-frac', str(val_frac)]
    if init_from is not None:
        init_path = _work_path(init_from)
        if not init_path.exists():
            raise FileNotFoundError(f'init_from does not exist: {init_path}')
        cmd += ['--init-from', str(init_path)]
    if unfreeze:
        if isinstance(unfreeze, (list, tuple, set)):
            unfreeze = ','.join(unfreeze)
        cmd += ['--unfreeze', str(unfreeze)]
    print('running:', ' '.join(shlex.quote(str(c)) for c in cmd), flush=True)
    # PYTHONUNBUFFERED=1 belt-and-braces alongside -u, in case the
    # child re-binds stdout to a logging handler that ignores -u.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    subprocess.run(cmd, cwd=WORK, check=True, env=env)

## 5. Experiment 1 — tiny-overfit (BLOCKING gate)

Train on 50 acted rows from the configured expert (currently `Ebi`), train==val. Should reach `train_loss < 0.1` and `top1 > 0.9` within ~150 epochs. **If this fails, STOP** — likely a mask / label-index / flatten / encoder-representation bug.

In [ ]:
import time
TS = time.strftime('%Y%m%d-%H%M%S')
OUT_OVERFIT = f'data/runs/pair_score/overfit_{PLAYER}_{TS}'

run_pair_score(
    out_dir=OUT_OVERFIT,
    max_rows=50,
    overfit=True,
    batch_size=32,
    lr=1e-3,
    epochs=150,
)

In [ ]:
import json
log = json.loads(open(f'{WORK}/{OUT_OVERFIT}/log.json').read())
last = log[-1]
print(f"epoch {last['epoch']}  tr_loss={last['train']['loss']:.4f}  tr_top1={last['train']['top1']:.3f}  val_top1={last['val']['top1']:.3f}")
if last['train']['loss'] > 0.1 or last['train']['top1'] < 0.9:
    print('\n⚠ overfit gate not cleared — investigate before Experiment 2.')
else:
    print('\n✓ overfit gate cleared — proceed to Experiment 2.')

## 6. Experiment 2 — small-real split (frozen encoders)

Up to 5000 acted rows from the expert's perspective, 80/20 train/val. Track top-1 / top-3 / top-5 vs the random-valid baseline. **Minimal success:** val top-1 ≥ 3× random. **Strong success:** val top-1 ≥ 0.30, top-3 ≥ 0.55. Encoders stay frozen — this is the head-only signal test.

In [ ]:
import time
TS = time.strftime('%Y%m%d-%H%M%S')
OUT_SMALL = f'data/runs/pair_score/small_{PLAYER}_{TS}'

run_pair_score(
    out_dir=OUT_SMALL,
    max_rows=5000,
    val_frac=0.2,
    batch_size=64,
    lr=1e-3,
    epochs=10,
)

In [ ]:
import json
log = json.loads(open(f'{WORK}/{OUT_SMALL}/log.json').read())
for e in log:
    v = e['val']
    rand = v.get('random_valid_top1', 0.0)
    margin = v['top1'] / max(rand, 1e-6)
    print(
        f"ep {e['epoch']:2d}  tr_loss={e['train']['loss']:.3f}  "
        f"val_loss={v['loss']:.3f}  val_top1={v['top1']:.3f}  "
        f"top3={v['top3']:.3f}  top5={v['top5']:.3f}  "
        f"rand={rand:.3f}  margin={margin:.1f}x"
    )
best = max(log, key=lambda e: e['val']['top1'])
print(
    f"\nbest val_top1={best['val']['top1']:.3f} "
    f"(epoch {best['epoch']}); rand={best['val'].get('random_valid_top1', 0.0):.3f}"
)

## 7. Experiment 3 — joint fine-tune (unfreeze CrossEntityAttention + PlanetEntityEncoder)

Resume from Exp 2's `pair_score_best.pt` and unfreeze **both** L2 (`CrossEntityAttention`) and L1 (`PlanetEntityEncoder`) alongside the head. Drops the `--max-rows` cap so training uses the full Ebi acted-snapshot pool (~40k rows, ~10× Experiment 2). Low LR (5e-5) for the joint fine-tune. The new ckpt saves cross + entity state so a deeper chain (e.g. `--unfreeze cross,entity,planet`) can resume from it.

In [ ]:
import time
TS = time.strftime('%Y%m%d-%H%M%S')
OUT_UNFREEZE = f'data/runs/pair_score/unfreeze_cross_entity_{PLAYER}_{TS}'

run_pair_score(
    out_dir=OUT_UNFREEZE,
    init_from=Path(OUT_SMALL) / 'pair_score_best.pt',
    unfreeze='cross,entity',
    max_rows=None,        # use every acted Ebi snapshot
    val_frac=0.2,
    batch_size=64,
    lr=5e-5,
    epochs=20,
)

## 8. Push results to GCS

In [ ]:
uploaded = []
for out in (OUT_OVERFIT, OUT_SMALL, globals().get('OUT_UNFREEZE')):
    if not out:
        continue
    out_path = _work_path(out)
    if not out_path.exists():
        print(f'skip missing run dir: {out_path}')
        continue
    subprocess.run(['gsutil', '-m', 'cp', '-r', str(out_path), f'{BUCKET}/runs/'], check=True)
    uploaded.append(f'{BUCKET}/runs/{out_path.name}')

print('uploaded:')
for url in uploaded:
    print(' ', url)

## 9. Decision tree

| Outcome | Read | Next step |
|---|---|---|
| Experiment 1 fails (no overfit) | label / mask / flatten / encoder bug | debug **before** any further training |
| Experiment 2 ≤ 1× random | encoder representation insufficient | revisit L1/L2 capacity, mask coverage, or pair feature **before** unfreeze |
| Experiment 2 ≥ 3× random, Experiment 3 improves val_top1 | unfreezing helped | chain `--unfreeze cross,entity --init-from <Exp 3 best>` |
| Experiment 2 ≥ 3× random, Experiment 3 worse / unstable | LR too high or encoder over-fitting small set | drop LR to 2e-5, halve epochs, or fall back to frozen-encoder Exp 2 best |
| Experiment 2 between | weak signal | try `--filter winner` on the expert, or another player (`flg`, `bowwowforeach`) before unfreezing |